We currently have two sets of functions for modifying fermionic Hamiltonians.

1. The function `process_fermionic_hamiltonian_to_remove_irrelevant_terms` in `utils_hamiltonian_ferm.py`
2. Various functions in  `utils_CSF_and_UCSF.py` (see imports below)


This notebook checks if these two functions return the same Hamiltonians. Since I (Smik) am not completely sure how to call the functions in `utils_CSF_and_UCSF.py` correctly yet, this notebook only checks a small subset of them. But even here, the result is that the `process_fermionic_hamiltonian_to_remove_irrelevant_terms` function removes a strict superset of terms compared with the corresponding function in `utils_CSF_and_UCSF.py`. This suggests that the analytic derivations of the effective Hamiltonians are not complete.

In [28]:
import sys
sys.path.append('../')

from openfermion import (
    normal_ordered,
    FermionOperator,
    get_sparse_operator as gso
)

import numpy as np
import pickle
from utils_CSF_and_UCSF import (
    orthogonal_transform_obt_tbt,
    obt_phys_spatial_to_spin,
    tbt_phys_spatial_to_spin,
    make_short_H_ferm_op,
    make_op_for_offdiag_U_1modiff,
    make_op_for_offdiag_U_2modiff,
    make_op_for_offdiag_UCSFiajb_UCSFiakb,
    make_op_for_offdiag_UCSFiajb_UCSFicjb,
    make_op_for_offdiag_UCSFia_UCSFiajb,
    make_op_for_offdiag_UCSFia_UCSFja,
    make_op_for_offdiag_UCSFia_UCSFib,
    make_op_for_offdiag_UCSFia_UCSFjb,
    make_op_for_offdiag_UCSFiajb_UCSFiakc,
    make_op_for_offdiag_UCSFiajb_UCSFkalb,
    make_op_for_offdiag_UCSFiajb_UCSFicjd,
    make_op_for_diagonal_U_space,
    make_op_for_diagonal_U_space_Stt,
    make_op_for_offdiag_UHF_UCSFia,
    make_op_for_offdiag_UHF_UCSFiajb
)
from utils_states import (
    convert_TZ_format_to_sparse_format,
    somos_to_seniority_config
)
from utils_hamiltonian_ferm import (
    process_fermionic_hamiltonian_to_remove_irrelevant_terms
)



In [29]:
# load states and Hamiltonian

filename = '../data/old_data1/h2o_data.dump'
with open(filename, 'rb') as f:
    (
    list_CSF,
    list_list_ia_CSF,
    list_list_theta_CSF,
    list_sym_CSF_vec,
    list_UCSF_tz,
    tz_states,
    somos_list,
    psi_GS_UCSF_smik,
    list_orb_rot,
    x_orbrot,
    Enuc,
    obt_spatial,
    tbt_spatial
    ) = pickle.load(f)

#Rotate orbitals 
if len(list_orb_rot) != 0:
    obt, tbt = orthogonal_transform_obt_tbt(x_orbrot,list_orb_rot,obt_spatial,tbt_spatial)
else:
    obt = obt_phys_spatial_to_spin(obt_spatial)
    tbt = tbt_phys_spatial_to_spin(tbt_spatial)

Nqubits = obt.shape[0]
dim     = 2 ** Nqubits
Norb    = Nqubits // 2
Nstates = len(tz_states)

Hferm        = make_short_H_ferm_op(0, obt, tbt)
configs      = [somos_to_seniority_config(somos, Norb) for somos in somos_list]
statevectors = [convert_TZ_format_to_sparse_format(dim, tz_state) for tz_state in tz_states]

In [30]:
i = 0
j = 0

bra        = statevectors[i]
bra_tz     = tz_states[i]
bra_somos  = somos_list[i]
bra_config = configs[i]

ket        = statevectors[j]
ket_tz     = tz_states[j]
ket_somos  = somos_list[j]
ket_config = configs[j]

print(f'''
    bra somos : {bra_somos}
    ket somos : {ket_somos}
''')

Hmod1 = make_op_for_diagonal_U_space(0, obt, tbt)
Hmod2 = process_fermionic_hamiltonian_to_remove_irrelevant_terms(Hferm, Nqubits, bra, ket_tz)

print(f'''
    Expectation Values (FULL) : {np.round(bra @ gso(Hferm     , Nqubits) @ ket.T, 8)[0,0]}
                       (TZ)   : {np.round(bra @ gso(sum(Hmod1), Nqubits) @ ket.T, 8)[0,0]}
                       (SP)   : {np.round(bra @ gso(Hmod2     , Nqubits) @ ket.T, 8)[0,0]}

    Number of Terms    (TZ)   : {len(normal_ordered(sum(Hmod1)).terms)}
                       (SP)   : {len(normal_ordered(Hmod2).terms)}

            
Difference :
{normal_ordered(sum(Hmod1) - Hmod2)}
''')


    bra somos : []
    ket somos : []


    Expectation Values (FULL) : (-83.78448781+0j)
                       (TZ)   : (-83.78448781+0j)
                       (SP)   : (-83.78448781+0j)

    Number of Terms    (TZ)   : 147
                       (SP)   : 135

            
Difference :
-0.0429160814423618 [1^ 0^ 3 2] +
-0.009912731893694373 [1^ 0^ 5 4] +
-0.021786222583613915 [1^ 0^ 7 6] +
-0.024344307211006663 [1^ 0^ 9 8] +
-0.02907972894554295 [1^ 0^ 11 10] +
-0.0200202961156386 [1^ 0^ 13 12] +
-0.04291608144236161 [3^ 2^ 1 0] +
-0.009912731893694371 [5^ 4^ 1 0] +
-0.02178622258361394 [7^ 6^ 1 0] +
-0.02434430721100667 [9^ 8^ 1 0] +
-0.029079728945542897 [11^ 10^ 1 0] +
-0.020020296115638613 [13^ 12^ 1 0]



In [ ]:
i = 0
j = 1

bra        = statevectors[i]
bra_tz     = tz_states[i]
bra_somos  = somos_list[i]
bra_config = configs[i]

ket        = statevectors[j]
ket_tz     = tz_states[j]
ket_somos  = somos_list[j]
ket_config = configs[j]

print(f'''
    bra somos : {bra_somos}
    ket somos : {ket_somos}
''')

list_CIS_pair = [[1,3]]


Hmod1 = make_op_for_offdiag_UHF_UCSFia(0, obt, tbt, None, list_CIS_pair)
Hmod2 = process_fermionic_hamiltonian_to_remove_irrelevant_terms(Hferm, Nqubits, bra, ket_tz)

print(f'''
    Expectation Values (FULL) : {np.round(bra @ gso(Hferm     , Nqubits) @ ket.T, 8)[0,0]}
                       (TZ)   : {np.round(bra @ gso(sum(Hmod1), Nqubits) @ ket.T, 8)[0,0]}
                       (SP)   : {np.round(bra @ gso(Hmod2     , Nqubits) @ ket.T, 8)[0,0]}

    Number of Terms    (TZ)   : {len(normal_ordered(sum(Hmod1)).terms)}
                       (SP)   : {len(normal_ordered(Hmod2).terms)}

            
Difference :
{normal_ordered(sum(Hmod1) - Hmod2)}
''')


    bra somos : []
    ket somos : [1, 3]


    Expectation Values (FULL) : (-0.1044808+0j)
                       (TZ)   : (-0.1044808+0j)
                       (SP)   : (-0.1044808+0j)

    Number of Terms    (TZ)   : 68
                       (SP)   : 64

            
Difference :
-0.014147938195797703 [1^ 0^ 6 3] +
0.014147938195797703 [1^ 0^ 7 2] +
-0.014147938195797703 [6^ 3^ 1 0] +
0.014147938195797703 [7^ 2^ 1 0]



In [38]:
i = 0
j = 2

bra        = statevectors[i]
bra_tz     = tz_states[i]
bra_somos  = somos_list[i]
bra_config = configs[i]

ket        = statevectors[j]
ket_tz     = tz_states[j]
ket_somos  = somos_list[j]
ket_config = configs[j]

print(f'''
    bra somos : {bra_somos}
    ket somos : {ket_somos}
''')

list_CIS_pair = [[1,5]]


Hmod1 = make_op_for_offdiag_UHF_UCSFia(0, obt, tbt, None, list_CIS_pair)
Hmod2 = process_fermionic_hamiltonian_to_remove_irrelevant_terms(Hferm, Nqubits, bra, ket_tz)

print(f'''
    Expectation Values (FULL) : {np.round(bra @ gso(Hferm     , Nqubits) @ ket.T, 8)[0,0]}
                       (TZ)   : {np.round(bra @ gso(sum(Hmod1), Nqubits) @ ket.T, 8)[0,0]}
                       (SP)   : {np.round(bra @ gso(Hmod2     , Nqubits) @ ket.T, 8)[0,0]}

    Number of Terms    (TZ)   : {len(normal_ordered(sum(Hmod1)).terms)}
                       (SP)   : {len(normal_ordered(Hmod2).terms)}

            
Difference :
{normal_ordered(sum(Hmod1) - Hmod2)}
''')


    bra somos : []
    ket somos : [1, 5]


    Expectation Values (FULL) : (-0.00328147+0j)
                       (TZ)   : (-0.00328147+0j)
                       (SP)   : (-0.00328147+0j)

    Number of Terms    (TZ)   : 68
                       (SP)   : 64

            
Difference :
0.02962397675282084 [1^ 0^ 10 3] +
-0.02962397675282084 [1^ 0^ 11 2] +
0.02962397675282084 [10^ 3^ 1 0] +
-0.02962397675282084 [11^ 2^ 1 0]



In [39]:
i = 1
j = 2

bra        = statevectors[i]
bra_tz     = tz_states[i]
bra_somos  = somos_list[i]
bra_config = configs[i]

ket        = statevectors[j]
ket_tz     = tz_states[j]
ket_somos  = somos_list[j]
ket_config = configs[j]

print(f'''
    bra somos : {bra_somos}
    ket somos : {ket_somos}
''')

list_CIS_pair = [[1,3], [1,5]]


Hmod1 = make_op_for_offdiag_UCSFia_UCSFib(obt, tbt, list_CIS_pair)
Hmod2 = process_fermionic_hamiltonian_to_remove_irrelevant_terms(Hferm, Nqubits, bra, ket_tz)

print(f'''
    Expectation Values (FULL) : {np.round(bra @ gso(Hferm     , Nqubits) @ ket.T, 8)[0,0]}
                       (TZ)   : {np.round(bra @ gso(sum(Hmod1), Nqubits) @ ket.T, 8)[0,0]}
                       (SP)   : {np.round(bra @ gso(Hmod2     , Nqubits) @ ket.T, 8)[0,0]}

    Number of Terms    (TZ)   : {len(normal_ordered(sum(Hmod1)).terms)}
                       (SP)   : {len(normal_ordered(Hmod2).terms)}

            
Difference :
{normal_ordered(sum(Hmod1) - Hmod2)}
''')


    bra somos : [1, 3]
    ket somos : [1, 5]


    Expectation Values (FULL) : (-0.03519481+0j)
                       (TZ)   : (-0.03519481+0j)
                       (SP)   : (-0.03519481+0j)

    Number of Terms    (TZ)   : 64
                       (SP)   : 60

            
Difference :
0.002334691155362705 [1^ 0^ 10 7] +
-0.002334691155362705 [1^ 0^ 11 6] +
0.002334691155362705 [10^ 7^ 1 0] +
-0.002334691155362705 [11^ 6^ 1 0]

